In [1]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

root
 |-- _c0: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- TP2: double (nullable = true)
 |-- TP3: double (nullable = true)
 |-- H1: double (nullable = true)
 |-- DV_pressure: double (nullable = true)
 |-- Reservoirs: double (nullable = true)
 |-- Oil_temperature: double (nullable = true)
 |-- Motor_current: double (nullable = true)
 |-- COMP: integer (nullable = true)
 |-- DV_electric: integer (nullable = true)
 |-- Towers: integer (nullable = true)
 |-- MPG: integer (nullable = true)
 |-- LPS: integer (nullable = true)
 |-- Pressure_switch: integer (nullable = true)
 |-- Oil_level: integer (nullable = true)
 |-- Caudal_impulses: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- dow: integer (nullable = true)

Tổng số bản ghi: 1516948


In [2]:
# Q1: THIẾT LẬP ĐƯỜNG CƠ SỞ VẬN HÀNH
q1_stats = spark.sql('''
    SELECT
        COMP,
        COUNT(*)                           AS so_ban_ghi,
        ROUND(AVG(Oil_temperature), 3)     AS nhiet_do_dau_tb,
        ROUND(STDDEV(Oil_temperature), 3)  AS nhiet_do_dau_lech_chuan,
        ROUND(MIN(Oil_temperature), 3)     AS nhiet_do_dau_min,
        ROUND(MAX(Oil_temperature), 3)     AS nhiet_do_dau_max,
        ROUND(AVG(Motor_current), 4)       AS dong_dien_tb,
        ROUND(STDDEV(Motor_current), 4)    AS dong_dien_lech_chuan,
        ROUND(AVG(TP3), 3)                 AS ap_suat_dau_ra_tb,
        ROUND(AVG(Reservoirs), 3)          AS ap_suat_binh_chua_tb
    FROM sensor
    GROUP BY COMP
    ORDER BY COMP
''')
print("EXECUTION PLAN")
q1_stats.explain(True)
# THỰC THI TRUY VẤN
print("Q1: THỐNG KÊ CHỈ SỐ VẬN HÀNH THEO TRẠNG THÁI MÁY NÉN")
df_q1 = q1_stats.toPandas()
try:
    display(df_q1)
except NameError:
    print(df_q1.to_string(index=False))

EXECUTION PLAN
== Parsed Logical Plan ==
'Sort ['COMP ASC NULLS FIRST], true
+- 'Aggregate ['COMP], ['COMP, 'COUNT(1) AS so_ban_ghi#44, 'ROUND('AVG('Oil_temperature), 3) AS nhiet_do_dau_tb#45, 'ROUND('STDDEV('Oil_temperature), 3) AS nhiet_do_dau_lech_chuan#46, 'ROUND('MIN('Oil_temperature), 3) AS nhiet_do_dau_min#47, 'ROUND('MAX('Oil_temperature), 3) AS nhiet_do_dau_max#48, 'ROUND('AVG('Motor_current), 4) AS dong_dien_tb#49, 'ROUND('STDDEV('Motor_current), 4) AS dong_dien_lech_chuan#50, 'ROUND('AVG('TP3), 3) AS ap_suat_dau_ra_tb#51, 'ROUND('AVG('Reservoirs), 3) AS ap_suat_binh_chua_tb#52]
   +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
COMP: int, so_ban_ghi: bigint, nhiet_do_dau_tb: double, nhiet_do_dau_lech_chuan: double, nhiet_do_dau_min: double, nhiet_do_dau_max: double, dong_dien_tb: double, dong_dien_lech_chuan: double, ap_suat_dau_ra_tb: double, ap_suat_binh_chua_tb: double
Sort [COMP#9 ASC NULLS FIRST], true
+- Aggregate [COMP#9], [COMP#9, count(1) AS 

,COMP,so_ban_ghi,nhiet_do_dau_tb,nhiet_do_dau_lech_chuan,nhiet_do_dau_min,nhiet_do_dau_max,dong_dien_tb,dong_dien_lech_chuan,ap_suat_dau_ra_tb,ap_suat_binh_chua_tb
0,0,247328,66.094,7.961,15.400,89.050,5.6038,1.0136,8.835,8.834
1,1,1269620,61.972,5.968,33.925,82.575,1.3579,1.7867,9.014,9.015
